In [17]:
import json
import os
import re
from bs4 import BeautifulSoup

In [6]:

#Läs in JSON-filen jag skapade  
with open("matematik.json", "r", encoding="utf-8") as f:
    data = json.load(f)
    subject = data.get("subject", {})
    chunks = []

# Skriv ut nycklar för att dubbelkolla struktur
print(data.keys())
print(subject.keys())

dict_keys(['apiPublisher', 'apiName', 'apiVersion', 'apiReleased', 'apiStatus', 'apiDocumentation', 'dataOrigin', 'calledMethod', 'totalElements', 'processingTime', 'startedCaching', 'codeParam', 'dateParam', 'subject'])
dict_keys(['centralCntHeading', 'centralContents', 'code', 'description', 'designation', 'gradeScale', 'knowledgeReqsHeading', 'knowledgeRequirements', 'modifiedDate', 'name', 'nameHeading', 'purpose', 'purposeHeading', 'schoolTypes', 'skolfsAndring', 'skolfsGrund', 'startDate', 'typeOfSyllabus', 'version', 'versionInfo'])


In [ ]:
centralt = subject.get('centralContents', []) 
for section in centralt: 
    print(section.get('heading', ''), '\n', section.get('text', '')[:100], '\n---') #Visa första 100 tecken för översikt/snabbkoll

 
 <h3>I årskurs 1–3</h3><h4>Taluppfattning och tals användning</h4> <ul> <li>Naturliga tal och deras e 
---
 
 <h3>I årskurs 4–6</h3><h4>Taluppfattning och tals användning</h4> <ul> <li>Rationella tal, däribland 
---
 
 <h3>I årskurs 7–9</h3><h4>Taluppfattning och tals användning</h4> <ul> <li>Reella tal och deras egen 
---


In [12]:
#Grundläggande metadata
subject_name = subject.get("name", "Okänt ämne")
subject_code = subject.get("code", "Okänd kod")
subject_version = subject.get("version", "Okänd version")

In [ ]:
#Funktion för att ta bort HTML-taggar och returnera ren text
def clean_html(html_text):
    if not html_text:
        return ""

    soup = BeautifulSoup(str(html_text), "html.parser")
    text = soup.get_text(separator=" ", strip=True)
    return text

In [19]:
#Funktion för att dela upp text utifrån h4-taggar 
def split_by_h4(html_text):
    parts = re.split(r'(<h4>.*?</h4>)', html_text)
    results = []
    current_heading = None

    for part in parts:
        part = part.strip()
        if not part:
            continue

        if part.startswith("<h4>"):
            current_heading = clean_html(part)
        else:
            if current_heading:
                text = clean_html(part)
                if text:
                    results.append({
                        "subheading": current_heading,
                        "text": text
                    })
    return results

In [14]:
# 1. Syfte
syfte_text = clean_html(subject.get("purpose"))
if syfte_text:
    chunks.append({
        "subject": subject_name,
        "subject_code": subject_code,
        "version": subject_version,
        "section": "syfte",
        "year": None,
        "heading": "Syfte",
        "text": syfte_text
    })

In [20]:
# 2. Centralt innehåll
central_contents = subject.get("centralContents", [])
print(f"Hittade {len(central_contents)} centralt innehåll-objekt")

for i, item in enumerate(central_contents):
    year = item.get("year")
    html_text = item.get("text", "")

    split_parts = split_by_h4(html_text)

    for part in split_parts:
        chunks.append({
            "subject": subject_name,
            "subject_code": subject_code,
            "version": subject_version,
            "section": "centralt_innehall",
            "year": year,
            "heading": f"{part['subheading']} ({year})",
            "text": part["text"]
        })


Hittade 3 centralt innehåll-objekt


In [16]:
# 3. Kunskapskrav / betygskriterier
knowledge_reqs = subject.get("knowledgeRequirements", [])
print(f"Hittade {len(knowledge_reqs)} kunskapskrav-objekt")
for i, item in enumerate(knowledge_reqs):
    text = clean_html(item.get("text"))
    year = item.get("year")
    
    if text:
        chunks.append({
            "subject": subject_name,
            "subject_code": subject_code,
            "version": subject_version,
            "section": "kunskapskrav",
            "year": year,
            "heading": f"Kunskapskrav {year}",
            "text": text
        })

Hittade 11 kunskapskrav-objekt


In [ ]:
#Spara resultatet 
os.makedirs("data", exist_ok=True)

with open("data/matematik_chunks.json", "w", encoding="utf-8") as f:
    json.dump(chunks, f, ensure_ascii=False, indent=2)

print(f"Klart! Skapade {len(chunks)} chunkar.")